# DuckDB Exercise: In-Process Analytical Database

**DuckDB** is an in-process OLAP (Online Analytical Processing) database. Unlike client-server databases (PostgreSQL, MySQL), DuckDB runs *inside* your Python process — no server to start, no ports to configure. It excels at fast analytical queries on large datasets directly from files (CSV, Parquet, JSON) or Pandas/Polars DataFrames.

### Topics covered
1. Installation & version check  
2. In-memory vs. persistent databases  
3. Creating tables and loading data  
4. Querying CSV and JSON files directly  
5. Pandas integration (zero-copy queries)  
6. Advanced SQL: window functions, CTEs, PIVOT  
7. Exporting data (Parquet, CSV)  
8. Performance: DuckDB vs. pandas groupby  

---
## 1. Installation

DuckDB is a pure Python package with no external dependencies — install it with a single `pip` command.

In [ ]:
# Install DuckDB (run once; skip if already installed)
%pip install duckdb --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import duckdb
print(f"DuckDB version: {duckdb.__version__}")

---
## 2. In-Memory vs. Persistent Databases

| Mode | How to open | Use case |
|------|-------------|----------|
| **In-memory** | `duckdb.connect()` or `duckdb.connect(':memory:')` | Temporary analysis, testing |
| **Persistent** | `duckdb.connect('mydb.duckdb')` | Save tables between sessions |

In [ ]:
# --- In-memory connection (default) ---
con = duckdb.connect()          # all data lives in RAM, gone when con closes

# Quick sanity check
result = con.execute("SELECT 'Hello, DuckDB!' AS greeting, 42 AS answer").fetchone()
print(result)

In [ ]:
# --- Persistent connection ---
# Writes a file called 'bioinformatics.duckdb' in the current directory.
pcon = duckdb.connect('bioinformatics.duckdb')
pcon.execute("CREATE TABLE IF NOT EXISTS notes (id INTEGER, note TEXT)")
pcon.execute("INSERT INTO notes VALUES (1, 'DuckDB persists between sessions')")
pcon.close()   # flush to disk

# Re-open and verify
pcon = duckdb.connect('bioinformatics.duckdb')
print(pcon.execute("SELECT * FROM notes").fetchall())
pcon.close()

---
## 3. Creating Tables and Loading Data

We will build a small genomics-flavored dataset: genes, samples, and expression values.

In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()   # fresh in-memory DB for the rest of the notebook

# ---- genes table ----
con.execute("""
    CREATE TABLE genes (
        gene_id   VARCHAR PRIMARY KEY,
        symbol    VARCHAR,
        chrom     VARCHAR,
        start_pos INTEGER,
        end_pos   INTEGER,
        biotype   VARCHAR
    )
""")

con.executemany("""
    INSERT INTO genes VALUES (?, ?, ?, ?, ?  , ?)
""", [
    ('ENSG001', 'BRCA1', 'chr17', 43044295, 43125364, 'protein_coding'),
    ('ENSG002', 'TP53',  'chr17', 7668421,  7687490,  'protein_coding'),
    ('ENSG003', 'EGFR',  'chr7',  55019017, 55211628, 'protein_coding'),
    ('ENSG004', 'MYC',   'chr8',  127735434,127742951,'protein_coding'),
    ('ENSG005', 'PTEN',  'chr10', 89623195, 89728532, 'protein_coding'),
    ('ENSG006', 'LINC01',  'chr1',  1000000,  1050000,  'lncRNA'),
    ('ENSG007', 'MIR21',   'chr17', 57918627, 57918698, 'miRNA'),
])

print("genes table:")
con.execute("SELECT * FROM genes").df()

In [ ]:
# ---- samples table ----
con.execute("""
    CREATE TABLE samples (
        sample_id  VARCHAR PRIMARY KEY,
        patient_id VARCHAR,
        tissue     VARCHAR,
        condition  VARCHAR,
        age        INTEGER
    )
""")

con.executemany("INSERT INTO samples VALUES (?,?,?,?,?)", [
    ('S01','P001','breast','tumor',  52),
    ('S02','P001','breast','normal', 52),
    ('S03','P002','breast','tumor',  47),
    ('S04','P002','breast','normal', 47),
    ('S05','P003','lung',  'tumor',  65),
    ('S06','P003','lung',  'normal', 65),
])

# ---- expression table (TPM values) ----
np.random.seed(42)
gene_ids   = ['ENSG001','ENSG002','ENSG003','ENSG004','ENSG005','ENSG006','ENSG007']
sample_ids = ['S01','S02','S03','S04','S05','S06']

rows = []
for gid in gene_ids:
    for sid in sample_ids:
        base = np.random.lognormal(mean=3, sigma=1.5)
        # inflate expression in tumor samples for BRCA1, TP53
        if gid in ('ENSG001','ENSG002') and sid in ('S01','S03','S05'):
            base *= 4
        rows.append((gid, sid, round(float(base), 4)))

con.execute("""
    CREATE TABLE expression (
        gene_id   VARCHAR REFERENCES genes(gene_id),
        sample_id VARCHAR REFERENCES samples(sample_id),
        tpm       DOUBLE
    )
""")
con.executemany("INSERT INTO expression VALUES (?,?,?)", rows)

print(f"expression rows: {con.execute('SELECT COUNT(*) FROM expression').fetchone()[0]}")

---
## 4. Basic SQL Queries

In [ ]:
# SELECT with filter and ORDER
con.execute("""
    SELECT symbol, chrom, end_pos - start_pos AS gene_length_bp
    FROM   genes
    WHERE  biotype = 'protein_coding'
    ORDER  BY gene_length_bp DESC
""").df()

In [ ]:
# JOIN: mean TPM per gene, per condition
con.execute("""
    SELECT g.symbol,
           s.condition,
           ROUND(AVG(e.tpm), 2)  AS mean_tpm,
           ROUND(STDDEV(e.tpm), 2) AS sd_tpm
    FROM   expression e
    JOIN   genes   g ON e.gene_id   = g.gene_id
    JOIN   samples s ON e.sample_id = s.sample_id
    GROUP  BY g.symbol, s.condition
    ORDER  BY g.symbol, s.condition
""").df()

In [ ]:
# HAVING: genes with mean tumor TPM > 100
con.execute("""
    SELECT g.symbol,
           ROUND(AVG(e.tpm), 2) AS mean_tumor_tpm
    FROM   expression e
    JOIN   genes   g ON e.gene_id   = g.gene_id
    JOIN   samples s ON e.sample_id = s.sample_id
    WHERE  s.condition = 'tumor'
    GROUP  BY g.symbol
    HAVING AVG(e.tpm) > 100
    ORDER  BY mean_tumor_tpm DESC
""").df()

---
## 5. Querying Files Directly (CSV & JSON)

DuckDB can query **CSV, JSON, and Parquet files directly** — no need to load them into a table first.

In [ ]:
import csv, json, pathlib

# Write a sample CSV file
csv_path = pathlib.Path('variants.csv')
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['variant_id','chrom','pos','ref','alt','gene','impact','af_gnomad'])
    w.writerows([
        ('rs1234','chr17',43094692,'A','T','BRCA1','HIGH',  0.0001),
        ('rs5678','chr17',43063902,'G','C','BRCA1','MODERATE',0.0012),
        ('rs9101','chr17',7674220, 'C','T','TP53', 'HIGH',  0.0003),
        ('rs1121','chr7', 55152337,'T','A','EGFR', 'MODERATE',0.0021),
        ('rs1314','chr17',7669609, 'G','A','TP53', 'LOW',   0.0150),
        ('rs1516','chr8', 127736062,'C','G','MYC', 'MODIFIER',0.0005),
    ])

print(f"Wrote {csv_path}")

# Write a sample JSON file
json_path = pathlib.Path('pathways.json')
with open(json_path, 'w') as f:
    json.dump([
        {"pathway_id": "hsa04110", "name": "Cell cycle",         "genes": ["TP53","MYC","EGFR"]},
        {"pathway_id": "hsa03440", "name": "Homologous recombination", "genes": ["BRCA1","TP53"]},
        {"pathway_id": "hsa04012", "name": "ErbB signaling",     "genes": ["EGFR","MYC"]},
        {"pathway_id": "hsa04115", "name": "p53 signaling",      "genes": ["TP53","PTEN"]},
    ], f, indent=2)

print(f"Wrote {json_path}")

In [ ]:
# Query CSV directly — DuckDB auto-detects schema
con.execute("SELECT * FROM 'variants.csv' WHERE impact = 'HIGH'").df()

In [ ]:
# Aggregate directly on the file — no table creation needed!
con.execute("""
    SELECT gene,
           COUNT(*)              AS n_variants,
           AVG(af_gnomad)        AS mean_af,
           COUNT(*) FILTER (WHERE impact = 'HIGH') AS n_high_impact
    FROM   'variants.csv'
    GROUP  BY gene
    ORDER  BY n_variants DESC
""").df()

In [ ]:
# Query JSON directly
# DuckDB reads JSON arrays of objects automatically
con.execute("SELECT pathway_id, name FROM 'pathways.json'").df()

---
## 6. Pandas Integration

DuckDB can query Pandas DataFrames **by name** — no copy, no import step.

In [ ]:
# Create a Pandas DataFrame
clinical_df = pd.DataFrame({
    'patient_id': ['P001','P002','P003'],
    'stage':      ['II',  'III', 'I'  ],
    'her2':       [True,  False, True ],
    'er':         [True,  True,  False],
    'survival_months': [48, 36, 60]
})

# DuckDB can reference the DataFrame directly by its Python variable name!
result = con.execute("""
    SELECT c.patient_id,
           c.stage,
           c.her2,
           c.survival_months,
           s.tissue,
           s.condition
    FROM   clinical_df c               -- <-- plain Python variable name
    JOIN   samples     s ON c.patient_id = s.patient_id
    WHERE  s.condition = 'tumor'
    ORDER  BY c.patient_id
""").df()

result

In [ ]:
# Result comes back as a DataFrame — chain with pandas if needed
expr_df = con.execute("""
    SELECT g.symbol, s.condition, e.tpm
    FROM   expression e
    JOIN   genes g   ON e.gene_id   = g.gene_id
    JOIN   samples s ON e.sample_id = s.sample_id
""").df()

print(type(expr_df))
expr_df.head()

---
## 7. Advanced SQL Features

### 7a. Window Functions

In [ ]:
# Rank genes by expression within each sample
con.execute("""
    SELECT g.symbol,
           e.sample_id,
           ROUND(e.tpm, 2) AS tpm,
           RANK() OVER (PARTITION BY e.sample_id ORDER BY e.tpm DESC) AS expr_rank
    FROM   expression e
    JOIN   genes g ON e.gene_id = g.gene_id
    QUALIFY expr_rank <= 3          -- DuckDB supports QUALIFY to filter window results
    ORDER  BY e.sample_id, expr_rank
""").df()

In [ ]:
# Rolling average TPM across chromosomal position (sorted genes)
con.execute("""
    SELECT g.symbol,
           g.chrom,
           g.start_pos,
           ROUND(AVG(e.tpm), 2) AS mean_tpm,
           ROUND(
               AVG(AVG(e.tpm)) OVER (
                   PARTITION BY g.chrom
                   ORDER BY g.start_pos
                   ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
               ), 2
           ) AS rolling_mean_tpm
    FROM   expression e
    JOIN   genes g ON e.gene_id = g.gene_id
    GROUP  BY g.symbol, g.chrom, g.start_pos
    ORDER  BY g.chrom, g.start_pos
""").df()

### 7b. Common Table Expressions (CTEs)

In [ ]:
# Fold-change: tumor / normal mean TPM per gene
con.execute("""
    WITH tumor AS (
        SELECT e.gene_id, AVG(e.tpm) AS mean_tpm
        FROM   expression e
        JOIN   samples s ON e.sample_id = s.sample_id
        WHERE  s.condition = 'tumor'
        GROUP  BY e.gene_id
    ),
    normal AS (
        SELECT e.gene_id, AVG(e.tpm) AS mean_tpm
        FROM   expression e
        JOIN   samples s ON e.sample_id = s.sample_id
        WHERE  s.condition = 'normal'
        GROUP  BY e.gene_id
    )
    SELECT g.symbol,
           ROUND(t.mean_tpm, 2)                              AS tumor_tpm,
           ROUND(n.mean_tpm, 2)                              AS normal_tpm,
           ROUND(t.mean_tpm / NULLIF(n.mean_tpm, 0), 3)     AS fold_change,
           ROUND(LOG2(t.mean_tpm / NULLIF(n.mean_tpm, 0)),3) AS log2fc
    FROM   tumor  t
    JOIN   normal n ON t.gene_id = n.gene_id
    JOIN   genes  g ON t.gene_id = g.gene_id
    ORDER  BY ABS(log2fc) DESC
""").df()

### 7c. PIVOT

In [ ]:
# Expression matrix: genes as rows, samples as columns
con.execute("""
    PIVOT (
        SELECT g.symbol, e.sample_id, ROUND(e.tpm, 1) AS tpm
        FROM   expression e
        JOIN   genes g ON e.gene_id = g.gene_id
    )
    ON sample_id
    USING SUM(tpm)
    GROUP BY symbol
""").df()

### 7d. List/Array Aggregation and UNNEST

In [ ]:
# Collect all sample IDs per gene where expression is "high" (TPM > 200)
con.execute("""
    SELECT g.symbol,
           LIST(e.sample_id ORDER BY e.sample_id) AS high_expr_samples,
           COUNT(*) AS n_samples
    FROM   expression e
    JOIN   genes g ON e.gene_id = g.gene_id
    WHERE  e.tpm > 200
    GROUP  BY g.symbol
    ORDER  BY n_samples DESC
""").df()

In [ ]:
# UNNEST: expand the JSON pathway→gene mapping into rows
# First, load pathways JSON into a table
con.execute("CREATE OR REPLACE TABLE pathways AS SELECT * FROM 'pathways.json'")

con.execute("""
    SELECT pathway_id, name, UNNEST(genes) AS gene_symbol
    FROM   pathways
    ORDER  BY pathway_id, gene_symbol
""").df()

---
## 8. Exporting Data

DuckDB can write query results directly to **CSV** or **Parquet** without going through pandas.

In [ ]:
# Export to CSV
con.execute("""
    COPY (
        SELECT g.symbol, s.condition, ROUND(AVG(e.tpm),2) AS mean_tpm
        FROM expression e
        JOIN genes g   ON e.gene_id   = g.gene_id
        JOIN samples s ON e.sample_id = s.sample_id
        GROUP BY g.symbol, s.condition
    ) TO 'mean_expression.csv' (HEADER, DELIMITER ',')
""")
print("Exported mean_expression.csv")

# Export to Parquet (columnar, compressed — ideal for large datasets)
con.execute("""
    COPY expression TO 'expression.parquet' (FORMAT PARQUET)
""")
print("Exported expression.parquet")

In [ ]:
# Query the Parquet file directly
con.execute("""
    SELECT COUNT(*) AS n_rows,
           ROUND(MIN(tpm),2)  AS min_tpm,
           ROUND(MAX(tpm),2)  AS max_tpm,
           ROUND(AVG(tpm),2)  AS avg_tpm
    FROM 'expression.parquet'
""").df()

---
## 9. Performance: DuckDB vs. pandas groupby

DuckDB's vectorized execution engine is significantly faster than pandas for large aggregations. Let's generate ~1 million rows and compare.

In [ ]:
import time

# Generate 1 million expression-like rows
N = 1_000_000
rng = np.random.default_rng(0)

big_df = pd.DataFrame({
    'gene_id':   rng.choice([f'G{i:04d}' for i in range(500)], N),
    'sample_id': rng.choice([f'S{i:03d}' for i in range(200)], N),
    'tpm':       rng.lognormal(3, 1.5, N),
    'condition': rng.choice(['tumor','normal'], N),
})

print(f"DataFrame shape: {big_df.shape}")
big_df.head(3)

In [ ]:
# --- pandas groupby ---
t0 = time.perf_counter()
pandas_result = (
    big_df
    .groupby(['gene_id','condition'])['tpm']
    .agg(['mean','std','count'])
    .reset_index()
)
pandas_time = time.perf_counter() - t0
print(f"pandas  : {pandas_time:.3f}s  |  result rows: {len(pandas_result)}")

In [ ]:
# --- DuckDB query on the same DataFrame ---
t0 = time.perf_counter()
duck_result = con.execute("""
    SELECT gene_id, condition,
           AVG(tpm)    AS mean,
           STDDEV(tpm) AS std,
           COUNT(*)    AS count
    FROM   big_df
    GROUP  BY gene_id, condition
""").df()
duck_time = time.perf_counter() - t0
print(f"DuckDB  : {duck_time:.3f}s  |  result rows: {len(duck_result)}")

speedup = pandas_time / duck_time
print(f"\nSpeedup : {speedup:.1f}x faster with DuckDB")

---
## 10. Bonus: DuckDB SQL Extensions Useful in Bioinformatics

### Regex extraction

In [ ]:
# Extract chromosome number from chrom column
con.execute("""
    SELECT symbol, chrom,
           REGEXP_EXTRACT(chrom, 'chr(\d+|X|Y)', 1) AS chrom_num
    FROM   genes
    ORDER  BY chrom_num
""").df()

### Descriptive statistics with SUMMARIZE

In [ ]:
# One-liner full statistical summary of any table or query
con.execute("SUMMARIZE expression").df()

### Show query plan

In [ ]:
# EXPLAIN shows the physical query plan — useful for optimization
plan = con.execute("""
    EXPLAIN
    SELECT g.symbol, COUNT(*) FROM expression e
    JOIN genes g ON e.gene_id = g.gene_id
    GROUP BY g.symbol
""").fetchall()

for row in plan:
    print(row[1])

---
## Summary

| Feature | DuckDB |
|---------|--------|
| Installation | `pip install duckdb` |
| In-memory DB | `duckdb.connect()` |
| Persistent DB | `duckdb.connect('file.duckdb')` |
| Query CSV/JSON/Parquet | `SELECT * FROM 'file.csv'` |
| Query Pandas DF | Use variable name directly in SQL |
| Results as DataFrame | `.df()` on any query |
| Export | `COPY (...) TO 'file.parquet'` |
| Unique SQL | `QUALIFY`, `PIVOT`, `SUMMARIZE`, `LIST()` |

**When to use DuckDB over pandas:**
- Aggregations, joins, or filters on DataFrames with > 100k rows  
- Reading subsets of large CSV/Parquet files without loading everything into memory  
- Complex analytical queries (window functions, CTEs) that are awkward in pandas  

**When to stay with pandas:**
- Row-by-row transformations or custom Python logic per row  
- Downstream ML pipelines expecting DataFrames  
- Simple operations on small data (< 10k rows) where overhead is negligible  